# Results prediction model
In questo notebook viene mostrata la pipeline che abbiamo adottato per la creazione di un modello che possa predire i risultati delle partite.

## Caricamento dataset

Come al solito, recuperiamo il dataset completo.

In [15]:
import pandas as pd
import numpy as np

path = "static/da-result/complete-result.csv"
dataset = pd.read_csv(path)

## Merge delle feature

Raggruppiamo le coppie di feature Home/Away in un unica feature data dal differenza tra feature home e feature away. Questo oltre a ridurre il numero di colonne, ci aiuta anche a ridurre la ridondanza e a semplificare il calcolo dei differenziali ai modelli. 

In [16]:
# Creo feature uniche
dataset["WinStreak"] = dataset["Home_WinStreak"] - dataset["Away_WinStreak"]
dataset["Z_Goals_Season"] = dataset["Z_Home_Goals_Season"] - dataset["Z_Away_Goals_Season"]
dataset["Z_Wins_Season"] = dataset["Z_Home_Wins_Season"] - dataset["Z_Away_Wins_Season"]
dataset["GoalOnShotRatio"] = dataset["GoalOnShotRatioHome"] - dataset["GoalOnShotRatioAway"]
dataset["PointToMatchRatio"] = dataset["PointToMatchRatioHome"] - dataset["PointToMatchRatioAway"]
dataset["Elo"] = dataset["HomeElo"] - dataset["AwayElo"]
dataset["Value"] = dataset["HomeValue"] - dataset["AwayValue"]

# Elimino le feature inutilizzate
dataset = dataset.drop(columns=["Home_WinStreak", "Away_WinStreak", "Z_Home_Goals_Season", "Z_Away_Goals_Season", "Z_Home_Wins_Season", "Z_Away_Wins_Season", "GoalOnShotRatioHome", "GoalOnShotRatioAway", "PointToMatchRatioHome", "PointToMatchRatioAway", "HomeElo", "AwayElo", "HomeValue", "AwayValue"])

dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Season,HomeAdvantage,WinStreak,Z_Goals_Season,Z_Wins_Season,GoalOnShotRatio,PointToMatchRatio,Elo,Value
0,2015-08-22,Lazio,Bologna,2,1,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,96050000.0
1,2015-08-22,Verona,Roma,1,1,D,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,-154100000.0
2,2015-08-23,Sassuolo,Napoli,2,1,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,-89400000.0
3,2015-08-23,Sampdoria,Carpi,5,2,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,37975000.0
4,2015-08-23,Palermo,Genoa,1,0,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,7600000.0


## Preparazione del target e split temporale

Procediamo:
- Mappando la colonna FTR in dati numerici (0 = Home, 1 = Draw, 2=Away)
- Ordinando cronologicamente il dataset
- Partizionandolo cronologicamente in set di training e set di test

In [17]:
from sklearn.metrics import accuracy_score, classification_report

# Mappo il target FTR in numeri
map = {'H': 0, 'D': 1, 'A': 2}
dataset['Numeric_target'] = dataset['FTR'].map(map)

# Ordino il dataset
dataset = dataset.sort_values(by='Date').reset_index(drop=True)

# Divido il traning set e test set in base alle stagioni (temporale)
train_mask = dataset['Season'] < '2024-2025'
test_mask = dataset['Season'] >= '2024-2025'

# Isolo il target per training e per test
Y_train = dataset.loc[train_mask, 'Numeric_target']
Y_test = dataset.loc[test_mask, 'Numeric_target']

X = dataset.drop(columns=['FTR'])
X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
X_train.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,Season,HomeAdvantage,WinStreak,Z_Goals_Season,Z_Wins_Season,GoalOnShotRatio,PointToMatchRatio,Elo,Value,Numeric_target
0,2015-08-22,Lazio,Bologna,2,1,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,96050000.0,0
1,2015-08-22,Verona,Roma,1,1,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,-154100000.0,1
2,2015-08-23,Sassuolo,Napoli,2,1,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,-89400000.0,0
3,2015-08-23,Sampdoria,Carpi,5,2,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,37975000.0,0
4,2015-08-23,Palermo,Genoa,1,0,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,7600000.0,0


### Random Baseline

Cominciamo osservando come un esperimento casuale si comporterebbe davanti alle partite del nostro test set (760). Questo passaggio lo utilizzamo per simulare lo scenario in cui un decisore (computer o un dado a tre facce), totalmente privo di dati storici o competenze calcistiche, tenti di indovinare l'esito dei match affidandosi al puro caso. 
L'output di questa simulazione fissa la nostra linea di partenza (random baseline) e diventa fondamentale per valutare il reale valore della ricerca.  

In [18]:
# Imposto il seed 
np.random.seed(42)

# Genero le predizioni totalmente casuali per le partite del test set
preds_casuali = np.random.choice([0, 1, 2], size=len(Y_test))

# Calcolo l'accuratezza del puro caso
accuratezza_puro_caso = accuracy_score(Y_test, preds_casuali)


print("\n==================================")
print("METRICHE DELL'ESPERIMENTO CASUALE")
print("==================================")
print(f"Accuratezza puro caso:    {accuratezza_puro_caso:.4f} ({accuratezza_puro_caso*100:.2f}%)")
print("\nReport dettagliato del Modello Casuale Puro:")
print(classification_report(Y_test, preds_casuali, target_names=['1', 'X', '2'], zero_division=0))


METRICHE DELL'ESPERIMENTO CASUALE
Accuratezza puro caso:    0.3501 (35.01%)

Report dettagliato del Modello Casuale Puro:
              precision    recall  f1-score   support

           1       0.43      0.40      0.41       291
           X       0.27      0.32      0.29       202
           2       0.33      0.32      0.33       241

    accuracy                           0.35       734
   macro avg       0.35      0.35      0.34       734
weighted avg       0.36      0.35      0.35       734



## Scelta delle feature
In questa sezione viene creata una classe che, dati in input le feature di interesse, utilizza l'interpolazione logistica per calcolare le metriche del modello, in funzione delle feautre che utilizziamo ingresso.

In [59]:
from pandas import DataFrame
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report

# Divido il traning set e test set in base alle stagioni (temporale)
test_mask = dataset['Season'] >= '2024-2025'
validation_mask = dataset['Season'] == '2023-2024'
train_mask = dataset['Season'] < '2023-2024'

X = dataset.drop(columns=['FTR'])
X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
X_eval = X.loc[validation_mask]

class LinearRegressionTester:

    __train_set: DataFrame
    __val_set: DataFrame
    __test_set: DataFrame

    def __init__(self, train_set: DataFrame, val_set: DataFrame, test_set: DataFrame):
        """
        :param train_set: Set di training, usato per addestrare il modello
        :param val_set: Set di validazione, usato per la selezione degli iperparametri
        :param test_set: Set di test, usato per la valutazione finale
        """
        self.__train_set = train_set
        self.__val_set = val_set
        self.__test_set = test_set

    def run_logistic_regression_baseline(
        self,
        target_col: str,
        baseline_features: list[str]
    ):
        """
        Esegue la baseline lineare (Regressione Logistica) con selezione
        dell'iperparametro C tramite il validation set, poi valuta sul test set.

        :param target_col: Il nome della colonna contenente le etichette target (Y)
        :param baseline_features: lista di feature con le quali viene valutato il modello
        """

        C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

        # Estrazione delle feature di interesse 
        X_train_features = self.__train_set[baseline_features]
        Y_train = self.__train_set[target_col]

        X_val_features = self.__val_set[baseline_features]
        Y_val = self.__val_set[target_col]

        X_test_features = self.__test_set[baseline_features]
        Y_test = self.__test_set[target_col]

        # Standardizzazione 
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_features)
        X_val_scaled = scaler.transform(X_val_features)
        X_test_scaled = scaler.transform(X_test_features)

        # Scetla degli iperparametri per evitare overfitting
        best_C = None
        best_val_score = -1.0
        best_model = None

        for C in C_values:
            model = LogisticRegression(
                solver='lbfgs',
                max_iter=500,
                random_state=42,
                class_weight="balanced",
                C=C
            )
            model.fit(X_train_scaled, Y_train)

            val_preds = model.predict(X_val_scaled)
            val_acc = accuracy_score(Y_val, val_preds)

            if val_acc > best_val_score:
                best_val_score = val_acc
                best_C = C
                best_model = model

        base_preds = best_model.predict(X_test_scaled)
        base_probs = best_model.predict_proba(X_test_scaled)

        print(f"\n=== Risultati sul Test Set (C={best_C}) ===")
        print(f"Accuratezza (Precision): {accuracy_score(Y_test, base_preds):.4f} ({accuracy_score(Y_test, base_preds)*100:.2f}%)")
        print(f"Log-Loss:    {log_loss(Y_test, base_probs):.4f}")
        print("\nReport:")
        print(classification_report(Y_test, base_preds, target_names=['1', 'X', '2'], zero_division=0))

        return best_model, scaler, best_C

## Scelta delle feature con regressione logistica
Adesso, tramite la classe appena creata, scegliamo delle feature e valutiamo l'impatto sulle metriche, eliminandone e aggiungendone di nuove, seguendo anche la heatmap del notebook feature-choice. Per prima cosa creaiamo un'istanza dalla classe, che successivamente andremo ad utilizzare.

In [60]:
ml = LinearRegressionTester(X_train, X_eval, X_test)

### Baseline con regressione logistica 

Realizziamo adesso una baseline basata sul modello della regressione logistica. Per farlo utilizziamo delle feature base da noi calcolate, come la striscia di vittorie, il rapporto punti/match ed altri. 

In [74]:
baseline = ["WinStreak","GoalOnShotRatio"] # Feature Base
ml.run_logistic_regression_baseline("Numeric_target", baseline)


=== Risultati sul Test Set (C=0.001) ===
Accuratezza (Precision): 0.3992 (39.92%)
Log-Loss:    1.0856

Report:
              precision    recall  f1-score   support

           1       0.44      0.59      0.50       291
           X       0.22      0.05      0.08       202
           2       0.38      0.46      0.41       241

    accuracy                           0.40       734
   macro avg       0.34      0.37      0.33       734
weighted avg       0.36      0.40      0.36       734



(LogisticRegression(C=0.001, class_weight='balanced', max_iter=500,
                    random_state=42),
 StandardScaler(),
 0.001)

si ha un ovvio miglioramento rispetto alla random baseline, sia per quanto riguarda l'accuratezza che per la precision e la recall. Sebbene la recall dei pareggi diminuisca

#### L'influenza dell'home advantage

Per capire se l'introduzione del **Fattore Campo (Home Advantage)** migliori effettivamente le performance del nostro modello di predizione, osserivamo le metriche una volta aggiunta questa feature

In [62]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["HomeAdvantage"])


=== Risultati sul Test Set (C=0.1) ===
Accuratezza (Precision): 0.4414 (44.14%)
Log-Loss:    1.0726

Report:
              precision    recall  f1-score   support

           1       0.51      0.50      0.50       291
           X       0.36      0.18      0.24       202
           2       0.41      0.59      0.49       241

    accuracy                           0.44       734
   macro avg       0.43      0.42      0.41       734
weighted avg       0.43      0.44      0.42       734



(LogisticRegression(C=0.1, class_weight='balanced', max_iter=500,
                    random_state=42),
 StandardScaler(),
 0.1)

Confrontando gli ultimi due report delle metriche ottenuti, notiamo che:

* **Vittoria in Casa:** La precision è salita nettamente dal **44% al 51%**. Il modello è diventato molto più affidabile quando pronostica la vittoria della squadra ospitante.
* **Pareggio:** Rimane l'esito più difficile da prevedere, ma si nota un leggero incremento della precision (da 22% a 36%) e dell'F1-score.
* **Vittoria in Trasferta:** Il recall è salito dal **46% al 59%**, segno che il modello riesce a catturare molte più vittorie esterne, probabilmente identificando meglio quando una squadra forte riesce a superare lo svantaggio del campo avversario.

> **Conclusione:** La feature *Home Advantage* ha un ruolo **significativo e positivo** nel modello. Non solo ha migliorato l'accuratezza generale del 4%, ma ha ridotto pure la log-loss, confermandosi una variabile importante per la predizione degli eventi sportivi.

#### PointToMatchRatio 
Facciamo un ulteriore passo avanti nel raffinamento del dataset introducendo una variabile essenziale per il rendimento delle squadre: il **Points Per Match (PPM) Ratio** (o *Media Punti per Partita*).

In [63]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["HomeAdvantage","PointToMatchRatio"])


=== Risultati sul Test Set (C=0.01) ===
Accuratezza (Precision): 0.4891 (48.91%)
Log-Loss:    1.0289

Report:
              precision    recall  f1-score   support

           1       0.53      0.56      0.55       291
           X       0.34      0.24      0.28       202
           2       0.52      0.61      0.56       241

    accuracy                           0.49       734
   macro avg       0.46      0.47      0.46       734
weighted avg       0.47      0.49      0.48       734



(LogisticRegression(C=0.01, class_weight='balanced', max_iter=500,
                    random_state=42),
 StandardScaler(),
 0.01)

L'integrazione del PPM Ratio ha influito positivamente su tutti e tre gli esiti:

* **Vittoria in Casa:** La precision sale ancora arrivando al **53%**, accoppiata a un ottimo recall del **55%**. Il modello ora intercetta benissimo le favorite in casa.
* **Pareggio:** Registra un incremento importante: la precision sale al **34%** (+4%) e l'F1-score passa da 0.24 a **0.28**.
* **Vittoria in Trasferta:** È l'aspetto che è migliorato maggiormente. La precision passa ad essere del **52%** (era al 41%) e il recall tocca il **61%**. Il PPM Ratio permette quindi al modello di capire quando una squadra ospite è statisticamente dominante, indipendentemente dal fattore campo.

> **Conclusione:** Il *Point to Match Ratio* si è rivelato la feature più importante inserita finora, registrando un miglioramento del 7% sull'accuratezza e una drastica riduzione della log-loss

#### Aggiunta dello z-score

Per fare un ulteriore salto di qualità rispetto alla baseline, introduciamo nel dataset le feature basate sullo **Z-Score**. Questa metrica esprime lo scostamento di una specifica caratteristica dalla media del campionato, misurato in unità di deviazioni standard.
L'introduzione dello Z-Score è fondamentale per catturare la **dinamica temporale** del campionato, risolvendo un limite intrinseco dei dati grezzi. Due valori numericamente identici possono infatti assumere significati opposti a seconda del momento in cui si verificano.


In [26]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["HomeAdvantage","PointToMatchRatio","Z_Wins_Season","Z_Goals_Season"])

Accuratezza (Precision): 0.5177 (51.77%)
Log-Loss:    1.0225

Report:
              precision    recall  f1-score   support

           1       0.57      0.60      0.59       291
           X       0.39      0.29      0.33       202
           2       0.53      0.61      0.57       241

    accuracy                           0.52       734
   macro avg       0.50      0.50      0.49       734
weighted avg       0.51      0.52      0.51       734



notiamo che l'accuratezza migliora ulteriormente del 2%, insieme ad altre metriche come la precision e la recall.

### Aggiunta dell'ELO
L'ultimo parametro che abbiamo deciso di utilizzare per l'allenamento del nostro modello è l'`elo`, ovvero il punteggio ottenuto prendendo ispirazione dall'omonimo sistema di valutazione degli scacchi. Variando le feature che utilizziamo, abbiamo notato che la configurazione migliore è la seguente, che porta, rispetto allo z-score, una piccola diminuzione della precision, ma la diminuzione della log-loss.

In [27]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["HomeAdvantage","Z_Wins_Season","Elo"])

Accuratezza (Precision): 0.5218 (52.18%)
Log-Loss:    0.9927

Report:
              precision    recall  f1-score   support

           1       0.57      0.62      0.59       291
           X       0.35      0.26      0.30       202
           2       0.56      0.63      0.59       241

    accuracy                           0.52       734
   macro avg       0.49      0.50      0.49       734
weighted avg       0.51      0.52      0.51       734



# Aggiunta del valore della rosa
Integriamo adesso l'ultima feature che abbiamo prodotto, ovvero il valore della rosa, indicato nel dataset come `Value`. Notiamo dalla heatmap che abbiamo prodotto nel notebook del feature choice che esso è correlato a molte altre feature, perciò escludiamole dalla valutazione del nsotro modello.

In [76]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["Value", "Z_Wins_Season"])


=== Risultati sul Test Set (C=0.01) ===
Accuratezza (Precision): 0.5450 (54.50%)
Log-Loss:    0.9944

Report:
              precision    recall  f1-score   support

           1       0.59      0.62      0.60       291
           X       0.41      0.32      0.36       202
           2       0.57      0.64      0.61       241

    accuracy                           0.54       734
   macro avg       0.52      0.53      0.52       734
weighted avg       0.53      0.54      0.54       734



(LogisticRegression(C=0.01, class_weight='balanced', max_iter=500,
                    random_state=42),
 StandardScaler(),
 0.01)

Notiamo in questo caso, principalmente, un grande miglioramento del parametro dell'**accuratezza**, che sfiora il valore del **55%**, decretando questa scelta come la migliore possibile. Notiamo anche un miglioramento della precision e della recall delle singole classi predette. Si riscontra un aumento importante nella classe dei pareggi (sia precision che recall).

### Modelli non lineari
In questa sezione utilizzeremo due modelli non lineari, ovvero la **discesa del gradiente** e la **random forest** per allenare il modello, con le intuizioni che abbiamo sviluppato osservando i risultati della regressione logistica. Per la discesa del gradiente utilizziamo la libreria **SGDClassifier** (lineare) e **GradientBoostingClassifier** di scikit-learn.

In [28]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import GradientBoostingClassifier


class NonLinearModels:

    __X_train_features: DataFrame
    __X_train_result: DataFrame
    __Y_test_features: DataFrame
    __Y_test_result: DataFrame

    def __init__(self, train_set: DataFrame, test_set: DataFrame, features: list[str], target: str):

        # Train
        self.__X_train_features = train_set[features]
        self.__X_train_result = train_set[target] 
        
        # Test
        self.__Y_test_features = test_set[features] 
        self.__Y_test_result = test_set[target]

    
    def __print_model_evaluation(self, predictions, probabilities):
        """
        Metodo che stampa i risultati del modello
        """

        # Metriche di Classificazione
        accuracy = accuracy_score(self.__Y_test_result, predictions)
        report = classification_report(self.__Y_test_result, predictions)
        loss_value = log_loss(self.__Y_test_result, probabilities)

        print("--- Modello di Classificazione Addestrato ---")
        print(f"Accuracy Globale: {accuracy * 100:.2f}%\n")
        print(f"Log Loss: {loss_value:.4f}")
        print("Dettaglio per ogni classe (0=Casa, 1=Pareggio, 2=Trasferta):")
        print(report)

    def gradient_descending(self):
        """
        Metodo che addestra un modello SGDRegressor utilizzando le feature passate
        """

        # Pipeline modello -> Scaler (per la normalizzazione) + regressore
        model = make_pipeline(
            StandardScaler(), 
            SGDClassifier(
                loss='log_loss', 
                max_iter=1000, 
                tol=1e-3, 
                random_state=42,
                penalty='l2' # Utilizziamo l2 di modo che non scarti le feature che ritiene "inutili"
            )
        )

        model.fit(self.__X_train_features, self.__X_train_result)

        predictions = model.predict(self.__Y_test_features)
        probabilities = model.predict_proba(self.__Y_test_features)
        self.__print_model_evaluation(predictions, probabilities)
    
    def gradient_boosting(self):
        """
        Metodo che addestra un modello GradientBoostingClassifier
        """

        model = GradientBoostingClassifier(
            n_estimators=200,       
            learning_rate=0.05,     
            max_depth=3,            
            subsample=0.8,          
            min_samples_leaf=20,    
            random_state=42
        )

        model.fit(self.__X_train_features, self.__X_train_result)

        predictions = model.predict(self.__Y_test_features)
        probabilities = model.predict_proba(self.__Y_test_features)

        self.__print_model_evaluation(predictions, probabilities)

In [29]:
nlm = NonLinearModels(X_train, X_test, ["WinStreak","HomeAdvantage", "GoalOnShotRatio","Z_Goals_Season", "Elo"], "Numeric_target")
nlm.gradient_descending()

--- Modello di Classificazione Addestrato ---
Accuracy Globale: 51.09%

Log Loss: 1.0156
Dettaglio per ogni classe (0=Casa, 1=Pareggio, 2=Trasferta):
              precision    recall  f1-score   support

           0       0.49      0.79      0.61       291
           1       0.00      0.00      0.00       202
           2       0.56      0.60      0.58       241

    accuracy                           0.51       734
   macro avg       0.35      0.46      0.39       734
weighted avg       0.38      0.51      0.43       734



In [30]:
nlm.gradient_boosting()

--- Modello di Classificazione Addestrato ---
Accuracy Globale: 53.27%

Log Loss: 1.0159
Dettaglio per ogni classe (0=Casa, 1=Pareggio, 2=Trasferta):
              precision    recall  f1-score   support

           0       0.51      0.75      0.61       291
           1       0.51      0.10      0.17       202
           2       0.57      0.63      0.60       241

    accuracy                           0.53       734
   macro avg       0.53      0.49      0.46       734
weighted avg       0.53      0.53      0.48       734

